# Heart Attack Risk Models
- Trains and compares different models like logistic regression, random forest, gradient boosting, etc in terms of heart attack prediction(HadHeartAttack column in the dataset). 
- Uses Stratified 5 Fold cross validation from sklearn, and tracks the models with ML flow.
- f2 scoring since its better to have a false positive than a false negative

In [45]:
import pandas as pd
import mlflow
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import make_scorer, fbeta_score

### Load Processed Data
80/20 split for train/test

In [46]:
# load train and test vars
X_train = pd.read_csv('X_train_processed.csv')
X_test = pd.read_csv('X_test_processed.csv')

Y_train = pd.read_csv('y_train.csv').squeeze() # .squeeze to convert from table to a series
Y_test = pd.read_csv('y_test.csv').squeeze()

print(X_train.shape)
print(X_test.shape)

print(Y_train.value_counts(normalize=True)) # % of 0s and 1s
print(Y_train.value_counts())# # of 0s and 1s


(196810, 50)
(49203, 50)
HadHeartAttack
0    0.945389
1    0.054611
Name: proportion, dtype: float64
HadHeartAttack
0    186062
1     10748
Name: count, dtype: int64


In [47]:
print(Y_train.value_counts(normalize=True))
print(Y_train.value_counts())

HadHeartAttack
0    0.945389
1    0.054611
Name: proportion, dtype: float64
HadHeartAttack
0    186062
1     10748
Name: count, dtype: int64


### Feature Sets
full means all 50 columns

lifestyle takes all but the ones that are conditions out

FEATURE_SETS makes it so users on our app can pick and choose what they input, and arent forced to input 50 things about themselves

- use feature importance later to pick which factors to keep

In [48]:
#chest scan and a couple other conditions can be the result of a heart problem, so group them together
CONDITION_COLS = [c for c in X_train.columns if "Had" in c or "ChestScan" in c]

# new model trained on just the cols added
FEATURE_SETS = {
    "full": X_train.columns.tolist(), # all
    "lifestyle": [c for c in X_train.columns if c not in CONDITION_COLS], # all but conditions
}

print(CONDITION_COLS)
print(f'Full data: {len(FEATURE_SETS["full"])}, Lifestyle only: {len(FEATURE_SETS["lifestyle"])}')

['numeric__HadAngina', 'numeric__HadStroke', 'numeric__HadAsthma', 'numeric__HadSkinCancer', 'numeric__HadCOPD', 'numeric__HadDepressiveDisorder', 'numeric__HadKidneyDisease', 'numeric__HadArthritis', 'numeric__ChestScan']
Full data: 50, Lifestyle only: 41


### MLflow + CV Setup
- MLflow tracks every run we do and you can access it using: --backend-store-uri sqlite:///mlflow.db
- stratified 5 fold cv on the training set, so the model is trained 5 times with 5 chunks at the same rate

In [49]:
# every run gets stored in a local DB
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("heart_attack")

<Experiment: artifact_location=('file:c:/Users/gabeh/Documents/Coding Projects/ML '
 'Project/CS4342-Project/mlruns/1'), creation_time=1790798208516, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1790798208516, lifecycle_stage='active', name='heart_attack', tags={}, trace_location=None, workspace='default'>

In [50]:
# 5 folds
# Each fold keeps the rate since its stratified
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# metrics
scoring = {
    "pr_auc": "average_precision",
    "roc_auc": "roc_auc",
    "recall": "recall",
    "precision": "precision",
    "f2": make_scorer(fbeta_score, beta=2),
}

### Baselines
Baseline logistic regression for HadHeartAttack and evaluation

In [51]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate

In [ ]:
results = [] # stores every model and feature set  with one row each

# evaluate functon makes it so we can pick a model, run cv on it, then log to MLFLOW and scores
def evaluate(model, name, feature_set):
    cols = FEATURE_SETS[feature_set]
    with mlflow.start_run(run_name=f"{name}_{feature_set}"):
        scores = cross_validate(model, X_train[cols], Y_train, cv=cv, scoring=scoring, n_jobs=-1)
        means = {m: scores[f"test_{m}"].mean() for m in scoring} # avg of the 5 folds
        mlflow.log_params({"model": name, "feature_set": feature_set})
        mlflow.log_metrics(means)
    row = {"model": name, "feature_set": feature_set}
    row.update(means)
    results.append(row)

In [ ]:
# baseline dummy classifier
# balanced makes sure false negatives are penalized more
# increase max iterations 10x so that we make sure it will converge
for fs in FEATURE_SETS:
    evaluate(DummyClassifier(strategy="most_frequent"), "dummy", fs)
    evaluate(LogisticRegression(class_weight="balanced", max_iter=1000), "logreg", fs)

pd.DataFrame(results).round(3)